# Bab 16 · Studi Kasus: Dari Data Mentah ke Model

**Notebook praktikum mahasiswa**  
Versi 2.0 · Pemrograman Komputer

- Memprediksi jumlah pesanan dengan fitur yang tersedia sebelum transaksi.
- Membagi data berdasarkan waktu dan menetapkan patokan.
- Menyusun pipeline data campuran serta melaporkan batas hasil.

### Petunjuk menjalankan sel · versi 2.0

Jalankan sel berurutan dari atas ke bawah. Setiap fungsi mandiri diletakkan pada sel tersendiri; sel pemanggilan atau pengujiannya menyusul setelah definisi. Setelah menyunting fungsi, jalankan ulang sel definisinya, lalu sel pengujiannya.

Sel persiapan dan fungsi pemeriksa cukup dijalankan; bagian yang Anda kerjakan ditandai **[ISI KODE]**. Metode yang membentuk satu kelas serta fungsi bersarang tetap disatukan karena merupakan satu kesatuan Python.

## Alur praktikum

**Duga → Jalankan → Selidiki → Isi kode → Periksa → Jelaskan**

Perkiraan waktu: 90–120 menit. Kerjakan berpasangan; tukar peran penulis kode dan pemeriksa setiap dua latihan.

| Penanda | Yang Anda kerjakan |
|---|---|
| [BACA] | Pahami konsep, kontrak fungsi, dan kasus batas. |
| [DUGA] | Tulis prediksi sebelum menjalankan contoh. |
| [COBA] | Jalankan contoh dan ubah satu hal untuk menyelidiki hasilnya. |
| [ISI KODE] | Lengkapi fungsi atau kelas; pertahankan nama dan parameternya. |
| [CEK OTOMATIS] | Jalankan pengujian yang terlihat, lalu gunakan pesannya untuk memperbaiki kode. |
| [REFLEKSI] | Jelaskan alasan dan bukti, bukan hanya menyalin keluaran. |

Impor file `.ipynb` ini ke notebook Python di Kaggle. Gunakan CPU; data kecil disediakan dalam notebook. Jalankan sel dari atas ke bawah. Pustaka yang diperlukan diimpor pada sel persiapan; tidak ada perintah instalasi atau unduhan.

`BELUM DIISI` adalah status normal pada notebook awal. Ganti `raise BelumDiisi()` dengan pekerjaan Anda. `LULUS` berarti memenuhi kasus uji yang tersedia, bukan bukti bahwa semua kemungkinan input sudah benar. Sel pengujian harus tetap utuh.

Jika kode berulang tanpa selesai, hentikan eksekusi, periksa batas perulangan, lalu jalankan ulang. Sebelum mengumpulkan, mulai ulang sesi Python dan jalankan seluruh sel agar hasil tidak bergantung pada variabel lama.

### Identitas

- Nama: …
- NIM: …
- Rekan diskusi: …
- Tanggal: …

**Persiapan dan pengaturan** · bagian 1 dari 12

In [ ]:
# [COBA] Jalankan sekali di awal; pemeriksaan tersedia untuk dibaca.
import math
import sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

**Definisi `BelumDiisi`** · bagian 2 dari 12

In [ ]:
class BelumDiisi(Exception):
    """Penanda latihan yang belum dikerjakan."""

**Definisi `sama`** · bagian 3 dari 12

In [ ]:
def sama(aktual, harapan):
    assert aktual == harapan, f"Diharapkan {harapan!r}; diperoleh {aktual!r}"

**Definisi `dekat`** · bagian 4 dari 12

In [ ]:
def dekat(aktual, harapan, atol=1e-8, rtol=1e-7):
    assert math.isclose(
        aktual, harapan, abs_tol=atol, rel_tol=rtol
    ), f"Diharapkan sekitar {harapan!r}; diperoleh {aktual!r}"

**Definisi `harus_galat`** · bagian 5 dari 12

In [ ]:
def harus_galat(jenis, panggil):
    try:
        panggil()
    except BelumDiisi:
        raise
    except jenis:
        return
    raise AssertionError(f"Seharusnya memunculkan {jenis.__name__}")

**Persiapan dan pengaturan** · bagian 6 dari 12

In [ ]:
DAFTAR_UJI = {}

**Definisi `cek`** · bagian 7 dari 12

In [ ]:
def cek(nomor, fungsi_uji, tampil=True):
    DAFTAR_UJI[nomor] = fungsi_uji
    try:
        fungsi_uji()
        status, pesan = "LULUS", "Semua kasus uji pada latihan ini sesuai."
    except BelumDiisi:
        status, pesan = (
            "BELUM DIISI",
            "Lengkapi sel [ISI KODE], jalankan, lalu ulangi pemeriksaan.",
        )
    except AssertionError as err:
        status, pesan = (
            "PERLU PERBAIKAN",
            str(err) or "Hasil belum sesuai kontrak latihan.",
        )
    except Exception as err:
        status, pesan = "GALAT", f"{type(err).__name__}: {err}"
    if tampil:
        print(f"Latihan {nomor} | {status}\n{pesan}")
    return status

**Definisi `rekap`** · bagian 8 dari 12

In [ ]:
def rekap():
    # Uji ulang fungsi terkini agar rekap tidak memakai status lama.
    hasil = {
        nomor: cek(nomor, uji, tampil=False)
        for nomor, uji in sorted(DAFTAR_UJI.items())
    }
    for nomor, status in hasil.items():
        print(f"  Latihan {nomor}: {status}")
    lulus = sum(s == "LULUS" for s in hasil.values())
    print(f"\nKemajuan uji otomatis: {lulus}/{JUMLAH_LATIHAN} latihan lulus.")
    print(
        "Refleksi, penjelasan, dan kualitas penyajian diperiksa bersama asisten."
    )
    return hasil

**Persiapan dan pengaturan** · bagian 9 dari 12

In [ ]:
print("Python:", sys.version.split()[0])
print("Siap. Jalankan notebook dari atas ke bawah.")
JUMLAH_LATIHAN = 4
import numpy as np

print("NumPy:", np.__version__)
import pandas as pd

print("pandas:", pd.__version__)
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score

**Persiapan dan pengaturan** · bagian 10 dari 12

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.compose import ColumnTransformer

print("scikit-learn:", sklearn.__version__)

**Definisi `siapkan_data_kasus`** · bagian 11 dari 12

In [ ]:
def siapkan_data_kasus():
    rng = np.random.default_rng(2026)
    n = 120
    tanggal = pd.date_range("2026-01-01", periods=n, freq="D")
    harga = rng.choice([20.0, 30.0, 40.0], n)
    promo = rng.integers(0, 2, n)
    kanal = rng.choice(["toko", "online"], n)
    jumlah = np.maximum(
        1,
        np.round(
            25
            - 0.3 * harga
            + 5 * promo
            + 2 * (kanal == "online")
            + rng.normal(0, 1.5, n)
        ),
    ).astype(int)
    df = pd.DataFrame(
        {
            "tanggal": tanggal,
            "harga": harga,
            "promo": promo,
            "kanal": kanal,
            "jumlah": jumlah,
        }
    )
    df["omzet"] = df.harga * df.jumlah
    return df

**Persiapan dan pengaturan** · bagian 12 dari 12

In [ ]:
data_kasus = siapkan_data_kasus()
FITUR = ["harga", "promo", "kanal"]

## [BACA] Konsep inti

Tujuan: menduga jumlah pesanan menggunakan harga yang ditawarkan, penanda promo, dan kanal. Data di bawah adalah simulasi, bukan transaksi nyata. Kolom omzet dihitung setelah jumlah diketahui, sehingga dilarang menjadi fitur. Evaluasi menggunakan periode sesudah data latih. Tetapkan MAE sebagai metrik utama; R² hanya pelengkap. Jangan memilih model berdasarkan holdout berulang kali.

## [DUGA] Prediksi sebelum eksekusi

Mengapa omzet tetap tidak boleh menjadi fitur meskipun korelasinya tinggi?

**Prediksi saya:** …

**Alasan:** …

In [ ]:
# [COBA]
print(data_kasus.head())
print("Periode:", data_kasus.tanggal.min(), "sampai", data_kasus.tanggal.max())
print(
    "Korelasi jumlah dan omzet:",
    data_kasus[["jumlah", "omzet"]].corr().iloc[0, 1],
)

**[REFLEKSI]** Apa perbedaan prediksi dan hasil? Ubah satu input pada contoh, tulis hasilnya, lalu jelaskan konsep yang ditunjukkan.

**Jawaban:** …

## Latihan 1 · Pemisahan waktu

**[ISI KODE]**

Buat `bagi_waktu(df, batas)` dari kolom tanggal bertipe datetime. Kembalikan salinan `(latih, uji)`, dengan tanggal < batas pada latih dan tanggal ≥ batas pada uji. Urutkan keduanya menaik menurut tanggal. Tolak salah satu bagian kosong dengan ValueError. Jangan mengubah input.

> Petunjuk: Gunakan dua mask pelengkap; perhatikan posisi tanggal yang tepat sama dengan batas.

In [ ]:
# [ISI KODE]
def bagi_waktu(df, batas):
    raise BelumDiisi()

**Definisi `uji_01`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_01():
    df = data_kasus.sample(frac=1, random_state=4)
    awal = df.copy(deep=True)
    lat, uji = bagi_waktu(df, "2026-04-01")
    sama((len(lat), len(uji)), (90, 30))
    assert (
        lat.tanggal.max() < uji.tanggal.min()
    ), "Periode latih mendahului uji."
    assert (
        lat.tanggal.is_monotonic_increasing
        and uji.tanggal.is_monotonic_increasing
    )
    sama(uji.tanggal.min(), pd.Timestamp("2026-04-01"))
    pd.testing.assert_frame_equal(df, awal)
    harus_galat(ValueError, lambda: bagi_waktu(df, "2030-01-01"))
    harus_galat(ValueError, lambda: bagi_waktu(df, "2020-01-01"))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(1, uji_01)

## Latihan 2 · Patokan median data latih

**[ISI KODE]**

Buat `patokan_median(y_latih, y_uji)` untuk array 1D berhingga tidak kosong. Kembalikan `(prediksi, mae)` dengan setiap prediksi bernilai median y_latih dan banyaknya sama dengan y_uji. Median tidak boleh dipelajari dari y_uji.

> Petunjuk: Gunakan np.full untuk membentuk prediksi konstan dengan panjang sesuai data uji.

In [ ]:
# [ISI KODE]
def patokan_median(y_latih, y_uji):
    raise BelumDiisi()

**Definisi `uji_02`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_02():
    p, m = patokan_median(np.array([1.0, 3.0, 9.0]), np.array([100.0, 102.0]))
    np.testing.assert_array_equal(p, [3, 3])
    dekat(m, 98)
    p, m = patokan_median(np.array([2.0, 4.0]), np.array([3.0]))
    np.testing.assert_array_equal(p, [3])
    dekat(m, 0)

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(2, uji_02)

## Latihan 3 · Pipeline campuran tanpa fitur bocor

**[ISI KODE]**

Buat `pipeline_pesanan()` yang mengembalikan Pipeline belum dilatih dengan langkah `pra` dan `model` (Ridge(alpha=1.0, solver="lsqr")). `pra` berupa ColumnTransformer: langkah `num` adalah Pipeline imputer median lalu StandardScaler untuk `["harga","promo"]`; langkah `kat` adalah OneHotEncoder(handle_unknown="ignore") untuk `["kanal"]`. Gunakan remainder="drop" agar tanggal, jumlah, dan omzet tidak masuk model. Kategori kanal tidak memuat nilai hilang; angka mungkin NaN.

> Petunjuk: Pilih kolom secara eksplisit di ColumnTransformer, lalu gabungkan dengan estimator.

In [ ]:
# [ISI KODE]
def pipeline_pesanan():
    raise BelumDiisi()

**Definisi `uji_03_struktur`** · bagian 1 dari 4

In [ ]:
def uji_03_struktur():
    p = pipeline_pesanan()
    assert isinstance(p, Pipeline) and list(p.named_steps) == ["pra", "model"]
    assert (
        isinstance(p["pra"], ColumnTransformer)
        and p["pra"].remainder == "drop"
    )
    assert isinstance(p["model"], Ridge)
    dekat(p["model"].alpha, 1)
    sama(p["model"].solver, "lsqr")
    tf = {nama: (obj, kolom) for nama, obj, kolom in p["pra"].transformers}
    sama(tf["num"][1], ["harga", "promo"])
    sama(tf["kat"][1], ["kanal"])
    assert isinstance(tf["num"][0], Pipeline)
    steps = list(tf["num"][0].named_steps.values())
    assert (
        len(steps) == 2
        and isinstance(steps[0], SimpleImputer)
        and steps[0].strategy == "median"
        and isinstance(steps[1], StandardScaler)
    )
    assert (
        isinstance(tf["kat"][0], OneHotEncoder)
        and tf["kat"][0].handle_unknown == "ignore"
    )

**Definisi `uji_03_prediksi`** · bagian 2 dari 4

In [ ]:
def uji_03_prediksi():
    p = pipeline_pesanan()
    lat = data_kasus.iloc[:90].copy()
    p.fit(lat, lat.jumlah)
    uji = data_kasus.iloc[90:93].copy()
    uji.loc[uji.index[0], "kanal"] = "reseller_baru"
    uji.loc[uji.index[1], "harga"] = np.nan
    pred = p.predict(uji)
    assert np.isfinite(pred).all()
    uji["jumlah"] = 99999
    uji["omzet"] = -99999
    np.testing.assert_allclose(p.predict(uji), pred)
    print("Prediksi dengan kategori baru dan nilai hilang:", pred)

**Definisi `uji_03`** · bagian 3 dari 4

In [ ]:
def uji_03():
    uji_03_struktur()
    uji_03_prediksi()

**Jalankan pemeriksaan** · bagian 4 dari 4

In [ ]:
cek(3, uji_03)

## Latihan 4 · Laporan evaluasi yang jujur

**[ISI KODE]**

Buat `laporkan(y, pred_model, pred_patokan)` untuk tiga array 1D berhingga dengan panjang sama ≥ 2; y tidak konstan. Kembalikan dict `n`, `mae_model`, `mae_patokan`, `r2_model`, `selisih_mae`, `lebih_baik`. Selisih = MAE patokan − MAE model; lebih_baik hanya True bila MAE model lebih kecil secara ketat. Sesudah lolos uji, jalankan eksperimen akhir pada sel tambahan dan tulis kesimpulan beserta batas data simulasi.

> Petunjuk: Tampilkan keunggulan sebagai selisih bertanda agar model yang lebih buruk tetap terlihat.

In [ ]:
# [ISI KODE]
def laporkan(y, pred_model, pred_patokan):
    raise BelumDiisi()

**Definisi `uji_04`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_04():
    h = laporkan(
        np.array([1.0, 2.0, 3.0]),
        np.array([1.0, 2.0, 3.0]),
        np.array([2.0, 2.0, 2.0]),
    )
    sama(h["n"], 3)
    dekat(h["mae_model"], 0)
    dekat(h["mae_patokan"], 2 / 3)
    dekat(h["r2_model"], 1)
    dekat(h["selisih_mae"], 2 / 3)
    assert h["lebih_baik"]
    seri = laporkan(
        np.array([1.0, 2.0, 3.0]),
        np.array([2.0, 2.0, 2.0]),
        np.array([2.0, 2.0, 2.0]),
    )
    assert not seri["lebih_baik"], "Hasil seri bukan perbaikan."
    buruk = laporkan(
        np.array([1.0, 2.0, 3.0]),
        np.array([9.0, 9.0, 9.0]),
        np.array([2.0, 2.0, 2.0]),
    )
    assert buruk["selisih_mae"] < 0 and not buruk["lebih_baik"]

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(4, uji_04)

## [COBA] Eksperimen akhir

Jalankan alur lengkap setelah seluruh fungsi selesai, lalu tafsirkan hasilnya pada refleksi akhir.

In [ ]:
# [COBA] Integrasi empat latihan; sel ini belum jalan penuh sebelum fungsi selesai.
try:
    latih, uji = bagi_waktu(data_kasus, "2026-04-01")
    dasar, _ = patokan_median(latih.jumlah.to_numpy(), uji.jumlah.to_numpy())
    pipa = pipeline_pesanan()
    pipa.fit(latih[FITUR], latih.jumlah)
    prediksi = pipa.predict(uji[FITUR])
    laporan = laporkan(uji.jumlah.to_numpy(), prediksi, dasar)
    print(laporan)
    print(
        "Tulis interpretasi MAE dalam satuan jumlah barang pada refleksi akhir."
    )
except BelumDiisi:
    print("BELUM DIISI: selesaikan empat fungsi dahulu.")

## [REFLEKSI] Refleksi akhir

Bagian ini membantu Anda merangkum pemahaman, mengenali kesulitan, dan menjelaskan alasan di balik kode. Tulis jawaban singkat berdasarkan percobaan Anda; bukan sekadar menyalin keluaran.

1. Pilih satu latihan. Jelaskan alur kode Anda dengan satu contoh input dan hasilnya.
2. Tuliskan satu kesalahan yang sempat terjadi, penyebabnya, dan cara memperbaikinya.
3. Usulkan satu kasus uji tambahan yang belum tercakup. Nyatakan hasil yang Anda harapkan dan alasannya.
4. Apa batas kesimpulan yang boleh dibuat dari hasil praktikum ini?

**Jawaban:** …

### Tantangan pengembangan

Tambahkan kasus uji usulan Anda pada sel di bawah. Pastikan kasus tersebut bisa membedakan implementasi benar dan satu kesalahan yang masuk akal. Diskusikan dengan asisten sebelum mengubah kontrak fungsi.

In [ ]:
# [ISI KODE OPSIONAL] Tambahkan eksperimen atau pengujian buatan Anda.
# Jelaskan harapan Anda pada komentar sebelum menjalankannya.

In [ ]:
# [CEK OTOMATIS] Uji ulang seluruh latihan yang sudah didaftarkan.
status_akhir = rekap()

## Sebelum mengumpulkan

- [ ] Identitas dan prediksi sudah diisi.
- [ ] Semua latihan sudah dikerjakan dan diperiksa dari sesi baru.
- [ ] refleksi akhir berisi penjelasan dengan bukti keluaran.
- [ ] Notebook disimpan dengan nama dan NIM; jangan hanya mengumpulkan HTML.

Rubrik diskusi: ketepatan kode 60%, penjelasan dan kasus batas 25%, keterbacaan serta kemampuan dijalankan ulang 15%. Rekap otomatis membantu belajar; penilaian akhir tetap memerlukan pemeriksaan asisten.

Rujukan: bab yang bersesuaian pada buku *Python untuk Machine Learning dan Data Science* dan modul praktikum. Latihan di notebook ini merupakan adaptasi terarah untuk praktikum, bukan seluruh soal akhir bab.